In [1]:
import json
import re
from glob import glob


outdir = "./english/error_handling/"

infiles = glob("./english/errors/*.umr")
all_errors = []
for file in infiles:
    all_errors.extend(open(file).readlines())

# all_errors = open("./english/error_handling/combined2.txt")

errors = {}
type_pattern = re.compile(r'\[L[0-9].*\]')
label_pattern = re.compile(r'\s[a-z]*-]*[a-z]*\]')
meta_pattern = re.compile(r'\[[^\]]*\]')
rel_pattern = re.compile(r'[^\s] \'.*\'\.')
head = 'minecraft-minecraft-4-11-sickles-B31-A23_1366'
head_mark = False
for line in all_errors:
    if line == '\n':
        head_mark = True
        continue
    if head_mark:
        head = line[:-1]
        head_mark = False
        continue
    type_search = re.search(type_pattern, line)
    if type_search:
        # label = re.search(label_pattern, type_search.group(0)).group(0)[1:-1]
        label = type_search.group(0)
        meta = re.search(meta_pattern, line).group(0)
        subtype = re.search(rel_pattern, line)
        if not subtype:
            subtype = "None"
        else:
            subtype = subtype.group(0)[3:-2]

        # if label not in errors:
        #     errors[label] = {subtype: [head + ": " + meta]}
        # else:
        #     if subtype not in errors[label]:
        #         errors[label][subtype] = [head + ": " + meta]
        #     else:
        #         errors[label][subtype].append(head + ": " + meta)
        
        if label not in errors:
            errors[label] = {subtype}
        else:
            errors[label].add(subtype)
for label in errors:
    errors[label] = list(errors[label])
open(outdir + "categories14.json", 'w').write(json.dumps(errors, indent=4))


20927

In [ ]:
from glob import glob


infiles = glob("./english/errors/*.umr")
outfile = "./english/error_handling/combined14.txt"
output = []
for file in infiles:
    output.append("\n")
    output.append(file.replace("./english/errors/", "").replace(".umr", ""))
    output.append("\n\n")
    output.extend(open(file).readlines())
open(outfile, 'w').writelines(output)

In [24]:
from glob import glob
import json
import re


infiles = glob("./english/errors/*.umr")
outfile = "./english/error_handling/block_format.json"
output = {}

line_pattern = re.compile(r'\[.*snt([0-9]+)\]:')

for file in infiles:
    lines = open(file).readlines()
    if len(lines) == 1:
        continue
    head = file.replace("./english/errors/", "").replace(".umr", "")
    curr_output = set()
    for line in lines:
        if '[L1 Format too-few-blocks]' in line:
            line_head = re.search(line_pattern, line).group(1)
            curr_output.add(f'.{line_head}')
    if curr_output:
        output[head] = curr_output

sent_pattern = re.compile(r'# meta-info :: sent_id = .*(\.[0-9]+) ::')

umr_files = glob("./english/release_ready/*.umr")
test = []
for file in umr_files:
    head = file.replace("./english/release_ready/", "").replace(".umr", "")
    if head not in output:
        continue
    outlines = []
    for line in open(file).readlines():
        id_match = re.match(sent_pattern, line)
        if id_match and id_match.group(1) in output[head]:
            outlines.append(line.replace('partial_conversion', 'not_in_release').replace('full_conversion', 'not_in_release'))
            output[head].remove(id_match.group(1))
            continue
        if id_match:
            test.append(id_match.group(1))
        outlines.append(line)
    open(file, 'w').writelines(outlines)
# open('./english/test.txt', 'w').writelines(test)
# open('./english/test.json', 'w').write(json.dumps(test, indent=4))
# open('./english/test_ids.txt', 'w').writelines(list(output))
# open('./english/test_ids.json', 'w').write(json.dumps(list(output), indent=4))
# print(len(output))
for key in output:
    if output[key]:
        print(key, output[key])


In [23]:
if set():
    print('yes')

In [3]:
from glob import glob
import json


infiles = glob("./english/errors/*.umr")
outfile = "./english/error_handling/combined14.json"
output = {}
for file in infiles:
    lines = open(file).readlines()
    if len(lines) == 1:
        continue
    output[file.replace("./english/errors/", "").replace(".umr", "")] = lines
open(outfile, 'w').write(json.dumps(output, indent=4))

12200763

In [18]:
import csv
import re
from glob import glob
from pprint import pprint


infiles = set(glob("./english/original_data/partial_conversion/**/**/*.txt", recursive=True))
outputs = []
id_pattern = re.compile(r"# ::id (.*?) ::")
var_pattern = re.compile(r"\((s[0-9]+[a-z][0-9]*) / cite-01")

for file in infiles:
    graph_marker = False
    describe_count = 0
    curr_output = {"var": "", "id": "", "graph": "", "sentence": ""}
    for line in open(file).readlines():
        if "# ::id" in line:
            try:
                curr_output["id"] = re.search(id_pattern, line).group(1)
            except:
                continue
            continue
        elif line.startswith("Words:   "):
            sent = line.replace("Words:  ", "").replace("  ", " ")
            curr_output["sentence"] = sent
            surface_count = 0
            for word in sent.split():
                if "citation" in word or "cite" in word or "citing" in word:
                    surface_count += 1
        elif line.startswith("# sentence level graph"):
            curr_graph = ""
            graph_marker = True
            continue
        elif graph_marker:
            if line == "\n":
                graph_marker = False
                curr_output["graph"] = curr_graph
                if describe_count:
                    if describe_count != surface_count:
                        outputs.append([curr_output["var"], curr_output["sentence"], curr_output["graph"], curr_output["id"]])
                        print(file, curr_output["id"])
                describe_count = 0
                curr_output = {"var": "", "id": "", "graph": "", "sentence": ""}
                continue
            if " cite-01" in line:
                describe_count += 1
                try:
                    curr_output["var"] = re.search(var_pattern, line).group(1)
                except:
                    print(line)
            curr_graph += line
with open('cite-01.tsv', 'w') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerows(outputs)


./english/original_data/partial_conversion/ldc/dfb/bolt-eng-DF-170-181105-8850361_0028.txt bolt-eng-DF-170-181105-8850361_0028.26


In [ ]:
import re
from glob import glob

def change_graph(lines, h_count, s_num):
    var_pattern = re.compile(r"\((s[0-9]+[a-z][0-9]*) / be-destined-for-91")
    var_map = {}
    new_lines = []
    for line in lines:
        if "be-destined-for-91" in line:
            var = re.search(var_pattern, line).group(1)
            var_map[var] = s_num + "h" + str(h_count)
            h_count += 1
            


h_pattern = re.compile(r"\(s[0-9]+(h[0-9]*) ")
infiles = glob("./english/original_data/**/*.txt", recursive=True)
for file in infiles:
    outfile = []
    h_count = 0
    d_count = 0
    g_mark = False
    curr_graph = []
    for line in open(file).readlines():
        if line.startswith("# :: snt"):
            s_num = line.replace("# :: snt", "").replace("\n", "")
        if line.startswith("# sentence level graph"):
            h_count = 0
            d_count = 0
            curr_graph = []
            d_mark = False
            g_mark = True
            continue
        if g_mark and line == "\n":
            curr_graph.append(line)
            if d_count:
                continue
            else:
                outfile.extend(curr_graph)
                g_mark = False
        h = re.search(h_pattern, line)
        if h is not None:
            if h == "h":
                h_count = max(h_count, 1)
            else:
                h_count = max(h_count, int(h.group(1)[1:]))
        if "be-destined-for-91" in line:
            d_count += 1
        #     var = re.search(var_pattern, line).group(1)
        #     h = re.search(h_pattern, line).group(1)
        #     line = line.replace("be-destined-for-91", "be-destined-for-91-h").replace(var, var + h)
        if g_mark:
            curr_graph.append(line)
        else:
            outfile.append(line)
    open(file, 'w').writelines(outfile)


In [3]:
import json

errors = [line for line in open("./english/error_handling/combined9.txt").readlines() if line.startswith("[")]
error_types = {key: 0 for key in json.loads(open("./english/error_handling/categories6.json").read())}
missing = []
for error in errors:
    for etype in error_types:
        if etype in error:
            error_types[etype] += 1
            break
for etype in error_types:
    print(etype, error_types[etype])
print(sum([error_types[etype] for etype in error_types]))

[L3 Sentence unknown-abstract-concept-ne] 9081
[L1 Format invalid-line] 47704
[L3 Sentence skipped-op-relation] 569
[L2 Sentence missing-node-definition] 41
[L3 Sentence unknown-relation] 933
[L3 Sentence repeated-relation] 333
[L3 Sentence unexpected-value] 247
[L1 Format too-few-blocks] 1698
[L3 Sentence wrong-incoming-name] 7
[L3 Sentence missing-incoming-name] 9
[L3 Sentence wrong-outgoing-name] 6
[L3 Document unknown-document-relation] 134
[L3 Document coref-entity-event-mismatch] 0
[L3 Document misplaced-document-relation] 4
[L3 Sentence missing-outgoing-name] 4
[L2 Metadata word-gloss-mismatch] 1
[L3 Document unknown-node-id] 25
[L2 Sentence non-unique-node-id] 0
[L2 Sentence extra-closing-bracket] 0
[L2 Alignment unknown-node-id] 0
[L0 Internal internal-error] 2
[L2 Document invalid-document-level] 15
60813


In [4]:
from glob import glob
import re


infiles = glob("./english/original_data/**/*.txt", recursive=True)
to_change = "# :: note: sentence not included in release and will not be added for future release"
to_change_2 = "# ::note: merged"
change_lines_1 = ["Index:       \n",
                "Words:   \n",
                "\n",
                "# sentence level graph:\n",]
change_lines_2 = ["\n",
                  "# alignment:\n"]
change_lines_3 = ["\n",
                  "# document level annotation:\n",
                  "\n",]
sentence_pattern = re.compile(r"# :: snt([0-9]+)")
for file in infiles:
    outputs = []
    for line in open(file).readlines():
        if line.startswith("# :: snt"):
            s_num = re.search(sentence_pattern, line).group(1)
        if to_change in line or to_change_2 in line:
            outputs.append(line)
            outputs.extend(change_lines_1)
            outputs.append(f"(s{s_num}u / umr-empty)\n")
            outputs.extend(change_lines_2)
            outputs.append(f"s{s_num}u: 0-0\n")
            outputs.extend(change_lines_2)
        else:
            outputs.append(line)
    open(file, 'w').writelines(outputs)

In [2]:
import penman
from pprint import pprint
from aspect import get_predicates

test = """(s21h / have-reason-91
      :ARG1 (s21a / and
                  :op1 (s21c / choose-01
                             :ARG0 (s21i / i)
                             :ARG1 (s21p / profession
                                         :mod (s21a2 / another)))
                  :op2 (s21l / learn-01
                             :ARG0 s21i
                             :ARG1 (s21p2 / pilot-01
                                          :ARG0 s21i
                                          :ARG1 (s21a3 / airplane)))))"""

graph = penman.decode(test)
pprint(get_predicates(graph))

[('s21h', 'have-reason-91'),
 ('s21c', 'choose-01'),
 ('s21l', 'learn-01'),
 ('s21p2', 'pilot-01')]


In [4]:
words = "Words: <Architect> For         this        one,        build       two         lines       of          three       touching    the         ground      with        one         block       gap         between     them        ".split()
print("".join(f"{i+1}: {word}\n" for i, word in enumerate(words[1:])))

1: <Architect>
2: For
3: this
4: one,
5: build
6: two
7: lines
8: of
9: three
10: touching
11: the
12: ground
13: with
14: one
15: block
16: gap
17: between
18: them



In [ ]:
import csv
import re
from glob import glob

infiles = glob('./english/release_ready/*')
output = []
outfile = "./umr_aspect/index.tsv"
id_pattern = re.compile(r'sent_id = (.*) :: type')
id_pattern2 = re.compile(r'sent_id = (.*)\n')
for file in infiles:
    for line in open(file).readlines():
        if line.startswith("# meta-info"):
            try:
                curr_id = re.search(id_pattern, line).group(1)
            except:
                try:
                    curr_id = re.search(id_pattern2, line).group(1)
                except:
                    print(line)
            continue
        if line.startswith("Words:"):
            curr_words = line.split()[1:]
            index = "".join([f"{i+1}: {word}\n" for i, word in enumerate(curr_words)])
            output.append([curr_id, index])
            continue
writer = csv.writer(open(outfile, 'w'), delimiter='\t')
writer.writerows(output)

In [8]:
import json
import csv
import penman


infile = json.loads(open('./umr_aspect/partial_conversion_2.json').read())
outlines = []
for graph_lines in infile:
    graph = penman.decode(graph_lines['graph'])
    vars = []
    targets = []
    for instance in graph.instances():
        if instance.source not in vars:
            vars.append(str(instance.source) + ": \n")
            targets.append(str(instance.source) + ": " + str(instance.target) + "\n")
    outlines.append([graph_lines['id'], "".join(vars), "".join(targets)])
csv.writer(open('./umr_aspect/partial_conversion_variables.tsv', 'w'), delimiter='\t').writerows(outlines)

# graph = penman.decode(infile[0]['graph'])
# for instance in graph.instances():
#     print(instance)

ignoring epigraph data for duplicate triple: ('s227p', ':mod', 's227c')
ignoring epigraph data for duplicate triple: ('s227p', ':mod', 's227c')
ignoring epigraph data for duplicate triple: ('s227p', ':mod', 's227c')
ignoring epigraph data for duplicate triple: ('s155l', ':ARG1', 's155b2')
ignoring epigraph data for duplicate triple: ('s155l', ':ARG1', 's155b2')
ignoring epigraph data for duplicate triple: ('s155l', ':ARG1', 's155b2')
ignoring epigraph data for duplicate triple: ('s155l', ':ARG1', 's155b2')
ignoring epigraph data for duplicate triple: ('s155l', ':ARG1', 's155b2')
ignoring epigraph data for duplicate triple: ('s155l', ':ARG1', 's155b2')
ignoring epigraph data for duplicate triple: ('s155l', ':ARG1', 's155b2')
ignoring epigraph data for duplicate triple: ('s155l', ':ARG1', 's155b2')
ignoring epigraph data for duplicate triple: ('s23c2', ':ARG1', 's23c')
ignoring epigraph data for duplicate triple: ('s23c2', ':ARG1', 's23c')
ignoring epigraph data for duplicate triple: ('s

In [2]:
from glob import glob
import json

infiles = glob("./english/release_ready/*.umr")
outdir = "./quantifiers/"

quantifiers = {
    "none": [],
    "some": [],
    "all": [],
    "many": [],
    "few": [],
    "several": [],
    "each": [],
    "every": [],
    "most": [],
    "no": [],
    "any": [],
    "both": [],
    "either": [],
    "lot": [],
    "lots": [],
    "any": [],
}
for file in infiles:
    curr_graph = []
    curr_quants = []
    for line in open(file).readlines():
        if line.startswith("################################################################################"):
            if curr_quants:
                for q in curr_quants:
                    quantifiers[q].append(curr_graph)
                curr_quants = []
            curr_graph = []
            curr_quants = []
        if line.startswith("Words: "):
            curr_words = line.split()[1:]
            for q in quantifiers:
                if q in curr_words:
                    curr_quants.append(q)
        curr_graph.append(line)

for q in quantifiers:
    open(outdir + q + ".txt", 'w').writelines("".join(graph) for graph in quantifiers[q])
json.dump(quantifiers, open(outdir + "quantifiers.json", 'w'), indent=4)    

In [4]:
print("".join([f"{q}, " for q in quantifiers]))

none, some, all, many, few, several, each, every, most, no, any, both, either, lot, lots, 


In [13]:
from glob import glob
import csv

workset_dict = {}
with open("./english/release_ready/directory_by_sentence.tsv", 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    next(reader)  # Skip header
    for row in reader:
        if row[0] not in workset_dict:
            # Initialize the list if the key does not exist
            workset_dict[row[0]] = {row[-1]}
        else:
            workset_dict[row[0]].add(row[-1])

count_dict = {key: 0 for key in workset_dict}
token_count_dict = {key: 0 for key in workset_dict}

infiles = glob("./english/release_ready/*.umr")
for file in infiles:
    token_count = 0
    count = 0
    for line in open(file).readlines():
        if line.startswith("################################################################################"):
            count += 1
        if "not_in_release" in line:
            count -= 1
        if line.startswith("Index: "):
            token_count += int(line.split()[-1])
    file = file.replace("./english/release_ready/", "")
    # print(file)
    for workset in workset_dict:
        if file in workset_dict[workset]:
            count_dict[workset] += count
            token_count_dict[workset] += token_count
            break
print(count_dict)
print(token_count_dict)

{'little prince': 1417, 'minecraft': 26218, 'ldc': 2313, 'umr': 2, 'pear_story': 141}
{'little prince': 19717, 'minecraft': 233468, 'ldc': 40257, 'umr': 18, 'pear_story': 1165}


In [10]:
test = "Index: 1     2     3     4     5     6     7     8     9     10    11    12    13    14    \n"
print(test.split())

['Index:', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14']
